# Ordered Logistic Regression Results for Adoption Predictors of Indigenous and Modern Knowledge in Rangeland Management Practices, Northern Kenya: Exploration with `mlcroissant`

This notebook provides an end-to-end guide for loading and exploring the [FAIR² Croissant dataset](https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json) using the `mlcroissant` library.

### Dataset Source

The dataset source is provided via a Croissant schema URL, which describes the structure, record sets, and fields of the data package.

In [ ]:
# Install `mlcroissant` if not already present
!pip install -U mlcroissant

## 1. Data Loading
Load the dataset metadata and records using the `mlcroissant` library.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the Croissant dataset schema URL
croissant_url = "https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json"

# Load the dataset - this parses metadata and tables as described in the Croissant schema
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata

print(f"Dataset name: {metadata.name if hasattr(metadata, 'name') else ''}\n")
print(f"Description: {metadata.description if hasattr(metadata, 'description') else ''}")

## 2. Data Overview
Review available record sets, fields, and their `@id`s.

All entities such as record sets, fields, and columns will be referenced **by their `@id`**.

Let's enumerate all record sets in the dataset and inspect their primary fields.

In [ ]:
# List available RecordSets by @id, with summary of constituent fields/columns @id
record_sets = dataset.record_sets

if not record_sets:
    print("No record sets found in this dataset.")
else:
    for rs in record_sets:
        print(f"RecordSet @id: {rs['@id']}")
        print(f"  Name: {rs.get('name', '[No name]')}")
        fields = rs.get('field', [])
        if isinstance(fields, dict):
            fields = [fields]
        if fields:
            print("  Fields/Columns @id:")
            for fld in fields:
                if isinstance(fld, dict):
                    print(f"    - {fld.get('@id')}: {fld.get('name', '[No name]')}")
                else:
                    print(f"    - {fld}")
        columns = rs.get('column', [])
        if isinstance(columns, dict):
            columns = [columns]
        if columns:
            print("  Columns @id:")
            for col in columns:
                if isinstance(col, dict):
                    print(f"    - {col.get('@id')}: {col.get('name', '[No name]')}")
                else:
                    print(f"    - {col}")
        print("-")

# As a demonstration, print the first record of each record set (if available)
for rs in record_sets:
    print(f"Fetching records for RecordSet @id: {rs['@id']}")
    try:
        records_iter = dataset.records(record_set=rs['@id'])
        first_record = next(records_iter, None)
        if first_record:
            print(f"  Sample record: {first_record}")
        else:
            print("  [No records found]")
    except Exception as e:
        print(f"  Error retrieving records: {e}")

## 3. Data Extraction

Now, let's extract data from **all available record sets** into pandas DataFrames for analysis, using each record set's `@id` for robust referencing.

To identify record set IDs (and field IDs) for downstream analysis, refer to the output from the previous step.

In [ ]:
# Gather all record set @id's for extraction
record_set_ids = [rs['@id'] for rs in dataset.record_sets]
dataframes = {}

for rsid in record_set_ids:
    # Use only the @id for referencing
    print(f"Loading record set: {rsid}")
    df = pd.DataFrame(dataset.records(record_set=rsid))
    if not df.empty:
        print(f"Loaded {len(df)} records. Columns: {df.columns.tolist()}")
    else:
        print(f"No data loaded for RecordSet @id: {rsid}")
    dataframes[rsid] = df

# For demonstration, print columns from the first available DataFrame with data
sample_rs = None
for rsid, df in dataframes.items():
    if not df.empty:
        sample_rs = rsid
        print(f"\nColumns in record set @id '{rsid}':\n{df.columns.tolist()}")
        display(df.head())
        break
if sample_rs is None:
    print("No dataframes with data available.")

## 4. Exploratory Data Analysis (EDA)

We'll select a **numeric field** from one of the extracted DataFrames and demonstrate:
- filtering records based on value criteria,
- normalizing a column,
- optionally grouping/aggregating if relevant fields exist.

All fields are referenced by their `@id`.

In [ ]:
# For illustration, pick the first DataFrame with at least one numeric column
import numpy as np

selected_rs = None
numeric_field = None

for rsid, df in dataframes.items():
    if not df.empty:
        numerics = df.select_dtypes(include=[np.number])
        if not numerics.empty:
            selected_rs = rsid
            numeric_field = numerics.columns[0]
            print(f"Selected RecordSet @id: {selected_rs}")
            print(f"Selected numeric field @id: {numeric_field}")
            break

if selected_rs is None or numeric_field is None:
    print("No numeric fields found to demonstrate EDA.")
else:
    # Filtering: retain records with value > threshold (example: 10)
    threshold = 10
    filtered_df = dataframes[selected_rs][dataframes[selected_rs][numeric_field] > threshold]
    print(f"Filtered records (where {numeric_field} > {threshold}): {len(filtered_df)} records")
    display(filtered_df.head())

    # Normalizing numeric field (z-score)
    norm_col = f"{numeric_field}_normalized"
    filtered_df[norm_col] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
    print(f"Normalized {numeric_field} (z-score) for filtered records:")
    display(filtered_df[[numeric_field, norm_col]].head())

    # Optionally group by another field if present
    possible_group_fields = [col for col in filtered_df.columns if col != numeric_field and filtered_df[col].dtype == object]
    group_field = possible_group_fields[0] if possible_group_fields else None
    if group_field:
        grouped_df = filtered_df.groupby(group_field).mean(numeric_only=True)
        print(f"Grouped by {group_field} (mean of numeric fields):")
        display(grouped_df.head())
    else:
        print("No categorical group field available for grouping.")

## 5. Visualization

Let's visualize the distribution of the selected numeric field (if available).

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Proceed if numeric_field is set and DataFrame exists
if selected_rs and numeric_field and not dataframes[selected_rs].empty:
    sns.histplot(dataframes[selected_rs][numeric_field].dropna(), kde=True)
    plt.title(f"Distribution of {numeric_field} in RecordSet {selected_rs}")
    plt.xlabel(numeric_field)
    plt.show()
else:
    print("No numeric field data available to plot.")

## 6. Conclusion

In this notebook, we demonstrated how to load and explore a [FAIR² Croissant dataset](https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json) using the `mlcroissant` library. 

- We loaded metadata and listed all record sets and their fields/columns by `@id`.
- We loaded available data into pandas DataFrames, and performed simple exploratory data analysis (filtering and normalization) based strictly on `@id` references.
- A basic visualization step illustrated how to examine numeric columns.

With this workflow, you can perform reproducible, schema-driven dataset analyses leveraging the FAIR² Croissant standard and powerful open-source Python tools.